# Практикум: статистика и A/B-тесты

Вторая половина курса. Здесь меньше задач, но каждая тяжелее: почти все требуют не вызвать функцию, а понять, какую функцию вызывать и что означает её вывод.

Проверка та же: `assert` внизу ячейки. Данные генерируются внутри ноутбука, сиды зафиксированы, поэтому ответы воспроизводимы.

**Порядок блоков**

1. Описательные статистики и выбросы
2. Случайность: почему выборка врёт
3. Доверительные интервалы
4. Проверка гипотез на средних
5. Проверка гипотез на долях
6. Ошибки, мощность, размер выборки
7. Как ломают A/B-тесты
8. Разбор Cookie Cats целиком

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

rng = np.random.default_rng(777)
np.set_printoptions(precision=4, suppress=True)

# Чеки интернет-магазина: логнормальное распределение — как в реальности
checks = np.round(rng.lognormal(mean=7.5, sigma=0.9, size=5000), 2)
# Несколько аномалий — оптовые закупки
checks = np.append(checks, [450000, 380000, 520000])

# Время на сайте, секунды
session = rng.gamma(shape=2.0, scale=90, size=3000)

print('чеков:', len(checks), '| среднее:', round(checks.mean(), 1), '| медиана:', round(np.median(checks), 1))

---
## Блок 1. Описательные статистики

Первое, что делают с числовой колонкой. И первое, где ошибаются.

- **Среднее** чувствительно к выбросам: один оптовый заказ на полмиллиона сдвинет его заметно.
- **Медиана** устойчива: она просто середина отсортированного ряда.
- **Перцентили** показывают форму: если 95-й перцентиль в разы больше медианы, у распределения длинный правый хвост.
- **Стандартное отклонение** — типичный разброс вокруг среднего.

Правило: если среднее сильно больше медианы, в отчёт идёт медиана, а в тест — либо медиана, либо усечённые данные.

### Задача 1. Среднее и медиана

Посчитай среднее и медиану `checks`, положи в `m` и `med`. В `gap` — во сколько раз среднее больше медианы.

<details><summary>Решение</summary>

```python
m, med = checks.mean(), np.median(checks)
gap = m / med
```

</details>

In [ ]:
m = ...
med = ...
gap = ...

# --- проверка, не менять ---
assert np.isclose(m, checks.mean()) and np.isclose(med, np.median(checks))
assert np.isclose(gap, m/med) and gap > 1
print("\u2713 сошлось")

### Задача 2. Перцентили

Посчитай 25-й, 50-й, 75-й и 95-й перцентили `checks` одним вызовом. Результат в `q` — массив из 4 чисел.

<details><summary>Подсказка</summary>

`np.percentile` принимает список процентов и возвращает массив.

</details>

<details><summary>Решение</summary>

```python
q = np.percentile(checks, [25, 50, 75, 95])
```

</details>

In [ ]:
q = ...

# --- проверка, не менять ---
assert len(q) == 4 and np.isclose(q[1], np.median(checks))
print("\u2713 сошлось")

### Задача 3. Поиск выбросов по IQR

Найди выбросы правилом полутора межквартильных размахов: всё выше `Q3 + 1.5*IQR`. Количество в `n_out`.

<details><summary>Подсказка</summary>

IQR — это расстояние между 25-м и 75-м перцентилями.

</details>

<details><summary>Решение</summary>

```python
q1, q3 = np.percentile(checks, [25, 75])
iqr = q3 - q1
n_out = (checks > q3 + 1.5*iqr).sum()
```

</details>

In [ ]:
n_out = ...

# --- проверка, не менять ---
q1, q3 = np.percentile(checks,[25,75]); iqr = q3-q1
assert n_out == (checks > q3 + 1.5*iqr).sum()
print("\u2713 сошлось")

### Задача 4. Что меняет удаление выбросов

Убери три самых больших значения и посчитай, на сколько процентов упадёт среднее. Результат в `drop_pct`.

<details><summary>Подсказка</summary>

Вот почему в A/B выбросы обрезают до начала расчётов, а не после.

</details>

<details><summary>Решение</summary>

```python
clean = np.sort(checks)[:-3]
drop_pct = (checks.mean() - clean.mean()) / checks.mean() * 100
```

</details>

In [ ]:
drop_pct = ...

# --- проверка, не менять ---
clean = np.sort(checks)[:-3]
assert np.isclose(drop_pct, (checks.mean()-clean.mean())/checks.mean()*100)
print("\u2713 сошлось")

---
## Блок 2. Случайность

Главная интуиция всей статистики: **любая выборка отличается от генеральной совокупности просто по случайности.** Задача тестов — отличить случайное колебание от настоящего эффекта.

Лучший способ это почувствовать — не читать, а симулировать.

### Задача 5. Выборка врёт

Возьми 50 случайных значений из `checks` и посчитай их среднее. Повтори 1000 раз через `rng.choice`. Разброс полученных средних (стандартное отклонение) положи в `se_sim`.

<details><summary>Подсказка</summary>

Именно этот разброс и называется стандартной ошибкой среднего.

</details>

<details><summary>Решение</summary>

```python
means = []
for _ in range(1000):
    s = rng.choice(checks, 50, replace=True)
    means.append(s.mean())
se_sim = np.std(means)
```

</details>

In [ ]:
means = []
for _ in range(1000):
    ...
se_sim = ...

# --- проверка, не менять ---
assert 'means' in dir() and len(means) == 1000
assert se_sim > 0 and abs(se_sim - np.std(means)) < 1e-9
print("\u2713 сошлось")

### Задача 6. Формула вместо симуляции

Стандартную ошибку можно посчитать формулой: стандартное отклонение, делённое на корень из размера выборки. Посчитай её для n=50 и положи в `se_formula`. Сравни с `se_sim`.

<details><summary>Подсказка</summary>

Числа близки — симуляция и формула говорят одно и то же.

</details>

<details><summary>Решение</summary>

```python
se_formula = checks.std(ddof=1) / np.sqrt(50)
```

</details>

In [ ]:
se_formula = ...

# --- проверка, не менять ---
assert np.isclose(se_formula, checks.std(ddof=1)/np.sqrt(50))
assert abs(se_formula - se_sim)/se_formula < 0.25
print("\u2713 сошлось")

### Задача 7. Как размер выборки гасит шум

Посчитай стандартную ошибку для n = 50, 200, 800, 3200. Результат в `se_by_n` — список из 4 чисел. Во сколько раз упала ошибка при росте n в 64 раза? В `ratio`.

<details><summary>Подсказка</summary>

Ошибка падает как корень из n: чтобы снизить её вдвое, выборку надо увеличить вчетверо. Отсюда растёт цена экспериментов.

</details>

<details><summary>Решение</summary>

```python
se_by_n = [checks.std(ddof=1)/np.sqrt(n) for n in [50,200,800,3200]]
ratio = se_by_n[0] / se_by_n[3]
```

</details>

In [ ]:
se_by_n = ...
ratio = ...

# --- проверка, не менять ---
assert len(se_by_n)==4 and np.isclose(ratio, se_by_n[0]/se_by_n[3])
assert 7 < ratio < 9
print("\u2713 сошлось")

---
## Блок 3. Доверительные интервалы

Интервал отвечает на вопрос «насколько точно мы знаем метрику». Формулировка, которую спрашивают: если повторить эксперимент много раз, 95% построенных таким способом интервалов накроют истинное значение.

Два способа: формула (для средних и долей) и бутстрап (для чего угодно).

### Задача 8. Интервал для среднего

Построй 95%-й доверительный интервал для среднего `session`. Границы в `lo` и `hi`.

<details><summary>Подсказка</summary>

1.96 — множитель для 95% в нормальном распределении.

</details>

<details><summary>Решение</summary>

```python
se = session.std(ddof=1) / np.sqrt(len(session))
lo, hi = session.mean() - 1.96*se, session.mean() + 1.96*se
```

</details>

In [ ]:
lo, hi = ..., ...

# --- проверка, не менять ---
se = session.std(ddof=1)/np.sqrt(len(session))
ref = (session.mean()-1.96*se, session.mean()+1.96*se)
assert np.isclose(lo, ref[0], rtol=.01) and np.isclose(hi, ref[1], rtol=.01)
print("\u2713 сошлось")

### Задача 9. Интервал для доли

Из 4000 посетителей купили 312. Построй 95%-й интервал для конверсии. Границы в `clo` и `chi`.

<details><summary>Подсказка</summary>

У доли своя формула ошибки: корень из p(1-p)/n.

</details>

<details><summary>Решение</summary>

```python
n, k = 4000, 312
p = k / n
se = np.sqrt(p*(1-p)/n)
clo, chi = p - 1.96*se, p + 1.96*se
```

</details>

In [ ]:
n, k = 4000, 312
clo, chi = ..., ...

# --- проверка, не менять ---
p = 312/4000; se = np.sqrt(p*(1-p)/4000)
assert np.isclose(clo, p-1.96*se, rtol=.01) and np.isclose(chi, p+1.96*se, rtol=.01)
print("\u2713 сошлось")

### Задача 10. Бутстрап-интервал

Построй 95%-й интервал для **медианы** `checks` бутстрапом на 2000 повторений. Границы в `blo` и `bhi`.

<details><summary>Подсказка</summary>

Для медианы формулы нет, а бутстрапу всё равно, какую метрику считать. В этом его ценность.

</details>

<details><summary>Решение</summary>

```python
boot = []
for _ in range(2000):
    s = rng.choice(checks, len(checks), replace=True)
    boot.append(np.median(s))
blo, bhi = np.percentile(boot, [2.5, 97.5])
```

</details>

In [ ]:
boot = []
for _ in range(2000):
    ...
blo, bhi = ..., ...

# --- проверка, не менять ---
assert len(boot)==2000 and blo < np.median(checks) < bhi
assert np.isclose(blo, np.percentile(boot,2.5)) and np.isclose(bhi, np.percentile(boot,97.5))
print("\u2713 сошлось")

---
## Блок 4. Гипотезы на средних

Нулевая гипотеза всегда одна и та же: разницы нет, всё что видим — случайность. Тест отвечает, насколько такие данные правдоподобны при этом предположении.

- `ttest_ind(a, b, equal_var=False)` — тест Уэлча, рабочая лошадка. Не требует равных дисперсий.
- `mannwhitneyu(a, b)` — когда распределение сильно скошено или выборка мала.
- Бутстрап — когда метрика нестандартная.

In [ ]:
# Две группы: контроль и тест, эффект +4% к времени на сайте
ctrl = rng.gamma(2.0, 90, 2400)
test = rng.gamma(2.0, 90*1.04, 2400)
print('ctrl:', round(ctrl.mean(),1), '| test:', round(test.mean(),1))

### Задача 11. Тест Уэлча

Сравни `ctrl` и `test` тестом Уэлча. Статистику положи в `t_stat`, p-value в `p_val`, а в `verdict` — строку `значимо` при p < 0.05, иначе `не значимо`.

<details><summary>Решение</summary>

```python
t_stat, p_val = stats.ttest_ind(ctrl, test, equal_var=False)
verdict = 'значимо' if p_val < 0.05 else 'не значимо'
```

</details>

In [ ]:
t_stat, p_val = ...
verdict = ...

# --- проверка, не менять ---
r = stats.ttest_ind(ctrl, test, equal_var=False)
assert np.isclose(p_val, r.pvalue)
assert verdict == ('значимо' if r.pvalue < .05 else 'не значимо')
print("\u2713 сошлось")

### Задача 12. Размер эффекта

p-value не говорит, насколько велика разница. Посчитай относительный прирост среднего в процентах — в `uplift`.

<details><summary>Подсказка</summary>

Значимый, но крошечный эффект бизнесу обычно не нужен. Всегда показывай обе цифры.

</details>

<details><summary>Решение</summary>

```python
uplift = (test.mean() - ctrl.mean()) / ctrl.mean() * 100
```

</details>

In [ ]:
uplift = ...

# --- проверка, не менять ---
assert np.isclose(uplift, (test.mean()-ctrl.mean())/ctrl.mean()*100)
print("\u2713 сошлось")

### Задача 13. Интервал для разницы

Бутстрапом на 2000 повторений построй 95%-й интервал для разницы средних (test минус ctrl). Границы в `dlo`, `dhi`, а в `zero_inside` — True, если ноль попал внутрь.

<details><summary>Подсказка</summary>

Если ноль внутри — разницу не доказали. Вывод тот же, что у p-value, но его можно показать бизнесу.

</details>

<details><summary>Решение</summary>

```python
diffs = []
for _ in range(2000):
    a = rng.choice(ctrl, len(ctrl), replace=True)
    b = rng.choice(test, len(test), replace=True)
    diffs.append(b.mean() - a.mean())
dlo, dhi = np.percentile(diffs, [2.5, 97.5])
zero_inside = dlo <= 0 <= dhi
```

</details>

In [ ]:
diffs = []
for _ in range(2000):
    ...
dlo, dhi = ..., ...
zero_inside = ...

# --- проверка, не менять ---
assert len(diffs)==2000
assert zero_inside == (dlo <= 0 <= dhi)
print("\u2713 сошлось")

### Задача 14. Непараметрический вариант

Сравни те же группы тестом Манна-Уитни. p-value в `mw_p`. В `same_conclusion` — True, если вывод про значимость совпал с тестом Уэлча.

<details><summary>Решение</summary>

```python
mw_stat, mw_p = stats.mannwhitneyu(ctrl, test)
same_conclusion = (mw_p < 0.05) == (p_val < 0.05)
```

</details>

In [ ]:
mw_p = ...
same_conclusion = ...

# --- проверка, не менять ---
r = stats.mannwhitneyu(ctrl, test)
assert np.isclose(mw_p, r.pvalue)
assert same_conclusion == ((mw_p < .05) == (p_val < .05))
print("\u2713 сошлось")

---
## Блок 5. Гипотезы на долях

Конверсия, retention, CTR — доли. У них своя арифметика: на входе не массив значений, а два числа на группу (сколько успехов, сколько всего).

### Задача 15. Хи-квадрат на конверсиях

Группа A: 6100 человек, 427 покупок. Группа B: 6050 человек, 484 покупки. Сравни конверсии через `chi2_contingency`. p-value в `conv_p`, конверсии в `cr_a` и `cr_b`.

<details><summary>Подсказка</summary>

Таблица сопряжённости строится как успехи и неуспехи, а не успехи и всего. Частая ошибка.

</details>

<details><summary>Решение</summary>

```python
table = np.array([[427, 6100-427], [484, 6050-484]])
chi2, conv_p, dof, exp = stats.chi2_contingency(table)
cr_a, cr_b = 427/6100, 484/6050
```

</details>

In [ ]:
table = ...
conv_p = ...
cr_a, cr_b = ..., ...

# --- проверка, не менять ---
t = np.array([[427, 6100-427],[484, 6050-484]])
ref = stats.chi2_contingency(t)[1]
assert np.isclose(conv_p, ref, rtol=.02)
assert np.isclose(cr_a, 427/6100) and np.isclose(cr_b, 484/6050)
print("\u2713 сошлось")

### Задача 16. Проверка деления групп (SRM)

Эксперимент разделил пользователей так: 45489 и 44700. Проверь через `chisquare`, похоже ли это на честные 50/50. p-value в `srm_p`, а в `srm_broken` — True, если p < 0.001.

<details><summary>Подсказка</summary>

Это первая проверка в любом анализе эксперимента. Если тут провал, остальное считать бессмысленно.

</details>

<details><summary>Решение</summary>

```python
srm_p = stats.chisquare([45489, 44700]).pvalue
srm_broken = srm_p < 0.001
```

</details>

In [ ]:
srm_p = ...
srm_broken = ...

# --- проверка, не менять ---
ref = stats.chisquare([45489, 44700]).pvalue
assert np.isclose(srm_p, ref)
assert srm_broken == (ref < 0.001)
print("\u2713 сошлось")

### Задача 17. Когда перекос становится опасным

Проверь SRM для деления 52000 и 48000 при тех же правилах. p-value в `srm_p2`, флаг в `srm_broken2`.

<details><summary>Подсказка</summary>

4% перекоса на ста тысячах — это уже сломанный эксперимент, хотя на глаз выглядит безобидно.

</details>

<details><summary>Решение</summary>

```python
srm_p2 = stats.chisquare([52000, 48000]).pvalue
srm_broken2 = srm_p2 < 0.001
```

</details>

In [ ]:
srm_p2 = ...
srm_broken2 = ...

# --- проверка, не менять ---
ref = stats.chisquare([52000, 48000]).pvalue
assert np.isclose(srm_p2, ref) and srm_broken2 == (ref < 0.001)
assert srm_broken2 is True or srm_broken2 == True
print("\u2713 сошлось")

---
## Блок 6. Ошибки, мощность, размер выборки

- **Ошибка I рода (альфа)** — объявили эффект, которого нет.
- **Ошибка II рода (бета)** — не заметили эффект, который есть.
- **Мощность** = 1 − бета, стандарт 80%.

Размер выборки считается до запуска из трёх чисел: альфа, мощность и минимальный эффект, ради которого стоит что-то менять.

### Задача 18. A/A-тест: ловим ложные срабатывания

Проведи 1000 A/A-тестов: обе группы по 500 из одного распределения `rng.normal(100, 15, 500)`, тест Уэлча. Доля p-value ниже 0.05 — в `fpr`. Должна выйти примерно 0.05.

<details><summary>Подсказка</summary>

Вот что такое альфа на практике: разницы нет, а в 5% прогонов тест её «находит». Это и есть цена порога 0.05.

</details>

<details><summary>Решение</summary>

```python
ps = []
for _ in range(1000):
    a = rng.normal(100, 15, 500)
    b = rng.normal(100, 15, 500)
    ps.append(stats.ttest_ind(a, b, equal_var=False).pvalue)
fpr = np.mean(np.array(ps) < 0.05)
```

</details>

In [ ]:
ps = []
for _ in range(1000):
    ...
fpr = ...

# --- проверка, не менять ---
assert len(ps)==1000 and 0.03 < fpr < 0.075
print("\u2713 сошлось")

### Задача 19. Мощность на симуляции

Теперь вторая группа со сдвигом: `rng.normal(103, 15, 500)`. Прогони 1000 раз и посчитай долю значимых результатов — это и есть мощность. В `power`.

<details><summary>Подсказка</summary>

Эффект есть всегда, но тест ловит его не в 100% случаев. Разница до 1.0 и есть ошибка II рода.

</details>

<details><summary>Решение</summary>

```python
ps2 = []
for _ in range(1000):
    a = rng.normal(100, 15, 500)
    b = rng.normal(103, 15, 500)
    ps2.append(stats.ttest_ind(a, b, equal_var=False).pvalue)
power = np.mean(np.array(ps2) < 0.05)
```

</details>

In [ ]:
ps2 = []
for _ in range(1000):
    ...
power = ...

# --- проверка, не менять ---
assert len(ps2)==1000 and 0.6 < power < 0.95
print("\u2713 сошлось")

### Задача 20. Расчёт размера выборки

Сколько нужно на группу, чтобы с мощностью 80% поймать прирост конверсии с 5% до 5.5%? Используй формулу через `(1.96+0.84)**2 * (p1*(1-p1)+p2*(1-p2)) / (p2-p1)**2`. Результат округли вверх в `n_needed`.

<details><summary>Подсказка</summary>

1.96 отвечает за альфу 5%, 0.84 — за мощность 80%. Число получится большим — и это нормальный повод сказать бизнесу, что эффект в 0.5 п.п. поймать дорого.

</details>

<details><summary>Решение</summary>

```python
p1, p2 = 0.05, 0.055
n_needed = int(np.ceil((1.96+0.84)**2 * (p1*(1-p1)+p2*(1-p2)) / (p2-p1)**2))
```

</details>

In [ ]:
p1, p2 = 0.05, 0.055
n_needed = ...

# --- проверка, не менять ---
p1, p2 = .05, .055
ref = int(np.ceil((1.96+0.84)**2 * (p1*(1-p1)+p2*(1-p2)) / (p2-p1)**2))
assert n_needed == ref
print("\u2713 сошлось")

### Задача 21. Цена маленького эффекта

Посчитай тем же способом размер выборки для прироста с 5% до 6%. Во сколько раз меньше, чем в прошлой задаче? В `times_less`.

<details><summary>Подсказка</summary>

Вдвое больший эффект требует вчетверо меньше людей. Поэтому первый вопрос перед экспериментом — какой минимальный эффект вообще интересен.

</details>

<details><summary>Решение</summary>

```python
p1, p2 = 0.05, 0.06
n_big = int(np.ceil((1.96+0.84)**2 * (p1*(1-p1)+p2*(1-p2)) / (p2-p1)**2))
times_less = n_needed / n_big
```

</details>

In [ ]:
n_big = ...
times_less = ...

# --- проверка, не менять ---
p1=.05; p2=.06
ref = int(np.ceil((1.96+0.84)**2 * (p1*(1-p1)+p2*(1-p2)) / (p2-p1)**2))
assert n_big == ref and np.isclose(times_less, n_needed/ref, rtol=.01)
assert times_less > 3
print("\u2713 сошлось")

---
## Блок 7. Как ломают A/B-тесты

Четыре способа получить неверный вывод, не допустив ни одной ошибки в формулах.

### Задача 22. Подглядывание

Симулируй 500 A/A-экспериментов по 2000 наблюдений на группу. В каждом проверяй p-value после 500, 1000, 1500 и 2000 наблюдений и считай эксперимент «успешным», если хотя бы в одной точке p < 0.05. Долю таких в `peek_fpr`.

<details><summary>Подсказка</summary>

Вышло заметно больше 5%, хотя разницы нет вообще. Вот цена привычки смотреть результат каждый день и останавливаться на зелёном.

</details>

<details><summary>Решение</summary>

```python
hits = 0
for _ in range(500):
    a = rng.normal(100, 15, 2000)
    b = rng.normal(100, 15, 2000)
    if any(stats.ttest_ind(a[:n], b[:n], equal_var=False).pvalue < 0.05 for n in [500,1000,1500,2000]):
        hits += 1
peek_fpr = hits / 500
```

</details>

In [ ]:
hits = 0
for _ in range(500):
    a = rng.normal(100, 15, 2000)
    b = rng.normal(100, 15, 2000)
    ...
peek_fpr = ...

# --- проверка, не менять ---
assert 0.08 < peek_fpr < 0.25
print("\u2713 сошлось")

### Задача 23. Множественные сравнения

Симулируй 1000 экспериментов, в каждом проверяя 10 независимых метрик без реального эффекта. Долю экспериментов, где хотя бы одна метрика «значима», положи в `multi_fpr`.

<details><summary>Подсказка</summary>

Около 40%. Проверил десять метрик — почти наверняка найдёшь «эффект». Лечится поправкой Бонферрони: порог делится на число метрик.

</details>

<details><summary>Решение</summary>

```python
wins = 0
for _ in range(1000):
    found = False
    for _ in range(10):
        a = rng.normal(0, 1, 400); b = rng.normal(0, 1, 400)
        if stats.ttest_ind(a, b, equal_var=False).pvalue < 0.05:
            found = True
    wins += found
multi_fpr = wins / 1000
```

</details>

In [ ]:
wins = 0
for _ in range(1000):
    ...
multi_fpr = ...

# --- проверка, не менять ---
assert 0.3 < multi_fpr < 0.55
print("\u2713 сошлось")

### Задача 24. Поправка Бонферрони

Повтори прошлую симуляцию, но считай метрику значимой при p < 0.05/10. Долю в `bonf_fpr`.

<details><summary>Подсказка</summary>

Вернулись примерно к 5%. Цена — упала мощность: мелкие настоящие эффекты теперь будут теряться.

</details>

<details><summary>Решение</summary>

```python
wins2 = 0
for _ in range(1000):
    found = False
    for _ in range(10):
        a = rng.normal(0, 1, 400); b = rng.normal(0, 1, 400)
        if stats.ttest_ind(a, b, equal_var=False).pvalue < 0.05/10:
            found = True
    wins2 += found
bonf_fpr = wins2 / 1000
```

</details>

In [ ]:
wins2 = 0
for _ in range(1000):
    ...
bonf_fpr = ...

# --- проверка, не менять ---
assert bonf_fpr < 0.1
print("\u2713 сошлось")

### Задача 25. Выброс ломает среднее

В группу B из 2000 наблюдений `rng.normal(100,15,2000)` добавь одно значение 9000. Посчитай p-value теста Уэлча против такой же группы A до и после добавления: `p_before` и `p_after`.

<details><summary>Подсказка</summary>

Одно значение из 2001 сдвигает вывод. Поэтому выбросы режут до теста, а решение об этом принимают заранее и записывают.

</details>

<details><summary>Решение</summary>

```python
a = rng.normal(100, 15, 2000)
b = rng.normal(100, 15, 2000)
p_before = stats.ttest_ind(a, b, equal_var=False).pvalue
b_dirty = np.append(b, 9000)
p_after = stats.ttest_ind(a, b_dirty, equal_var=False).pvalue
```

</details>

In [ ]:
a = rng.normal(100, 15, 2000)
b = rng.normal(100, 15, 2000)
p_before = ...
b_dirty = ...
p_after = ...

# --- проверка, не менять ---
assert p_after != p_before
assert len(b_dirty) == 2001
print("\u2713 сошлось")

---
## Блок 8. Cookie Cats целиком

Финальный блок — весь путь на настоящих данных. Скачай `cookie_cats.csv` и положи рядом с этим ноутбуком.

Порядок действий, который надо воспроизвести самому:

1. Загрузить, посмотреть размер, типы, пропуски, дубли по `userid`.
2. **SRM**: проверить деление групп хи-квадратом. Если сломано — дальше не идти.
3. Найти и обработать выбросы по `sum_gamerounds`. Решение зафиксировать: что удалил и почему.
4. Посчитать `retention_1` и `retention_7` по группам.
5. Сравнить доли: хи-квадрат или z-тест пропорций.
6. Построить бутстрап-интервал для разницы retention_7.
7. Написать вывод на языке решения: переносить ворота или нет, какой эффект, насколько уверены.

Ниже каркас. Заполняй сам, проверять эти ячейки некому — в этом и смысл финального блока.

In [ ]:
import pandas as pd
from scipy import stats

df = pd.read_csv('cookie_cats.csv')

# 1. разведка
print(df.shape)
print(df.dtypes)
print('дубли userid:', df.userid.duplicated().sum())

# 2. SRM
obs = df.version.value_counts().values
print('группы:', obs, '| SRM p =', stats.chisquare(obs).pvalue)

# 3. выбросы
# твой код

# 4. метрики по группам
# твой код

# 5. сравнение долей
# твой код

# 6. бутстрап-интервал для разницы
# твой код

# 7. вывод текстом


---
## Что дальше

Пройденный материал закрывает статистическую часть стажёрских собеседований. Остаётся то, что не выучивается из ноутбука:

- **Дизайн эксперимента словами.** «Хотим поменять кнопку оформления заказа» → какая метрика целевая, какие контрольные, сколько нужно пользователей, сколько крутим, что делаем при каждом исходе. Напиши три таких дизайна для выдуманных фич и дай кому-нибудь прочитать.
- **Разбор чужой ошибки.** Возьми любой публичный ноутбук по A/B на Kaggle и найди, что там сделано не так. Почти всегда найдётся: нет проверки SRM, нет обработки выбросов, t-тест на долях, вывод заканчивается на p-value.

Второе упражнение полезнее первого: искать дыры в готовом анализе — ровно то, чем аналитик занимается на ревью.